# End-to-End Data Preparation and Feature Engineering Using Pandas
## For an AI-Based Customer Analytics System

---
**Topic:** Pandas for Data Preprocessing in AI Applications  
**Role:** Data Engineer — Preparing raw customer data for the AI team

---

## 📦 Step 0 — Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ All libraries imported successfully!")
print(f"   Pandas version  : {pd.__version__}")
print(f"   NumPy  version  : {np.__version__}")

---
## 📂 Section 1 — Data Loading

In [ ]:
# ── Load all three datasets ──────────────────────────────────────────────────
customers   = pd.read_csv('customers.csv')
purchases   = pd.read_csv('purchases.csv')
membership  = pd.read_excel('membership.xlsx')

print("✅ Datasets loaded successfully!")
print(f"   customers  : {customers.shape[0]} rows × {customers.shape[1]} columns")
print(f"   purchases  : {purchases.shape[0]} rows × {purchases.shape[1]} columns")
print(f"   membership : {membership.shape[0]} rows × {membership.shape[1]} columns")

In [ ]:
# ── First 10 rows ────────────────────────────────────────────────────────────
print("=" * 60)
print("CUSTOMERS — First 10 rows")
print("=" * 60)
display(customers.head(10))

print("\n" + "=" * 60)
print("PURCHASES — First 10 rows")
print("=" * 60)
display(purchases.head(10))

print("\n" + "=" * 60)
print("MEMBERSHIP — First 10 rows")
print("=" * 60)
display(membership.head(10))

In [ ]:
# ── Last 10 rows ─────────────────────────────────────────────────────────────
print("CUSTOMERS — Last 10 rows")
display(customers.tail(10))

print("PURCHASES — Last 10 rows")
display(purchases.tail(10))

print("MEMBERSHIP — Last 10 rows")
display(membership.tail(10))

In [ ]:
# ── Shape, column names, dtypes ───────────────────────────────────────────────
for name, df in [('customers', customers), ('purchases', purchases), ('membership', membership)]:
    print(f"\n{'='*55}")
    print(f"  {name.upper()} — Info")
    print(f"{'='*55}")
    print(f"  Shape   : {df.shape}")
    print(f"  Columns : {list(df.columns)}")
    print()
    print(df.dtypes.rename('DType').to_frame())

In [ ]:
# ── Summary Statistics ────────────────────────────────────────────────────────
print("CUSTOMERS — Summary Statistics")
display(customers.describe(include='all'))

print("\nPURCHASES — Summary Statistics")
display(purchases.describe(include='all'))

print("\nMEMBERSHIP — Summary Statistics")
display(membership.describe(include='all'))

---
## 🧹 Section 2 — Data Cleaning

In [ ]:
# ── 2.1  Missing Values — Before Cleaning ─────────────────────────────────────
print("Missing Values BEFORE Cleaning")
print("=" * 40)

for name, df in [('customers', customers), ('purchases', purchases), ('membership', membership)]:
    miss = df.isnull().sum()
    miss = miss[miss > 0]
    print(f"\n{name.upper()}:")
    if miss.empty:
        print("  No missing values.")
    else:
        for col, cnt in miss.items():
            pct = cnt / len(df) * 100
            print(f"  {col:<18}: {cnt} missing  ({pct:.1f}%)")

In [ ]:
# ── 2.2  Fill Missing Values ──────────────────────────────────────────────────

# customers — Age → median, Income → mean, City → 'Unknown'
age_median    = customers['Age'].median()
income_mean   = customers['Income'].mean()

customers['Age'] = customers['Age'].fillna(age_median)
customers['Income'] = customers['Income'].fillna(income_mean)
customers['City'] = customers['City'].fillna('Unknown')

# purchases — Amount → median
amount_median = purchases['Amount'].median()
purchases['Amount'] = purchases['Amount'].fillna(amount_median)

print("✅ Missing values filled.")
print(f"   Age    filled with median : {age_median}")
print(f"   Income filled with mean   : {income_mean:.2f}")
print(f"   City   filled with        : 'Unknown'")
print(f"   Amount filled with median : {amount_median}")

In [ ]:
# ── 2.3  Missing Values — After Cleaning ──────────────────────────────────────
print("Missing Values AFTER Cleaning")
print("=" * 40)
for name, df in [('customers', customers), ('purchases', purchases)]:
    total = df.isnull().sum().sum()
    print(f"  {name.upper():<15}: {total} missing values remaining")

In [ ]:
# ── 2.4  Remove Duplicates ────────────────────────────────────────────────────
before_cust = len(customers)
before_purch = len(purchases)

customers = customers.drop_duplicates()
customers = customers.reset_index(drop=True)

purchases = purchases.drop_duplicates()
purchases = purchases.reset_index(drop=True)

removed_cust  = before_cust  - len(customers)
removed_purch = before_purch - len(purchases)

print("Duplicate Removal Report")
print("=" * 40)
print(f"  customers  : {removed_cust}  duplicates removed  ({before_cust} → {len(customers)} rows)")
print(f"  purchases  : {removed_purch} duplicates removed  ({before_purch} → {len(purchases)} rows)")

In [ ]:
# ── 2.5  Data Standardisation ─────────────────────────────────────────────────

# Standardize City  (lahore / LAHORE / lahore → Lahore)
customers['City']   = customers['City'].str.strip().str.title()

# Standardize Gender  (m/male/Female → M/F)
gender_map = {'m': 'M', 'male': 'M', 'M': 'M',
              'f': 'F', 'female': 'F', 'Female': 'F', 'F': 'F'}
customers['Gender'] = customers['Gender'].str.strip().map(
    lambda x: gender_map.get(x.lower() if isinstance(x, str) else x, x)
)

# Standardize MembershipType  (bronze / GOLD / gold → Bronze / Gold)
membership['MembershipType'] = membership['MembershipType'].str.strip().str.title()

print("✅ Standardisation complete.")
print("\n  Unique Cities  :", sorted(customers['City'].unique()))
print("  Unique Genders :", sorted(customers['Gender'].unique()))
print("  Membership types:", sorted(membership['MembershipType'].unique()))

In [ ]:
# ── 2.6  Outlier Detection & Handling ─────────────────────────────────────────

# Age outliers
age_outliers = customers[(customers['Age'] < 18) | (customers['Age'] > 70)]
print(f"Age outliers found    : {len(age_outliers)} rows")
display(age_outliers[['CustomerID','Name','Age']])

# Replace invalid ages with median
valid_age_median = customers[(customers['Age'] >= 18) & (customers['Age'] <= 70)]['Age'].median()
customers.loc[(customers['Age'] < 18) | (customers['Age'] > 70), 'Age'] = valid_age_median

# Negative Income
neg_income = customers[customers['Income'] < 0]
print(f"\nNegative Income rows  : {len(neg_income)}")
customers.loc[customers['Income'] < 0, 'Income'] = income_mean

# Quantity outliers in purchases
qty_outliers = purchases[purchases['Quantity'] <= 0]
print(f"\nInvalid Quantity rows : {len(qty_outliers)}")
display(qty_outliers[['PurchaseID','CustomerID','Quantity']])
purchases = purchases[purchases['Quantity'] > 0].reset_index(drop=True)
print(f"\nPurchases remaining after removing invalid quantities: {len(purchases)}")

print("\n✅ Outlier handling complete.")

In [ ]:
# ── 2.7  Cleaning Summary Report ──────────────────────────────────────────────
report = pd.DataFrame({
    'Check'      : ['Age missing filled','Income missing filled','City missing filled',
                    'Amount missing filled','Cust duplicates removed','Purch duplicates removed',
                    'City standardised','Gender standardised','Membership standardised',
                    'Age outliers fixed','Negative income fixed','Invalid quantity removed'],
    'Action'     : ['Filled with median','Filled with mean','Replaced with Unknown',
                    'Filled with median','Dropped','Dropped',
                    'Title case','M/F mapping','Title case',
                    'Replaced with valid median','Replaced with mean','Rows removed'],
    'Records'    : [2,3,0,2,removed_cust,removed_purch,
                    16,16,10,
                    len(age_outliers),len(neg_income),len(qty_outliers)]
})
print("\n📋 CLEANING SUMMARY REPORT")
display(report)

---
## 🔍 Section 3 — Data Exploration and Filtering

In [ ]:
# ── Filter 1: Customers earning above 80,000 ──────────────────────────────────
high_earners = customers[customers['Income'] > 80000]
print(f"Customers earning > 80,000  :  {len(high_earners)} found")
display(high_earners[['CustomerID','Name','Income','City']])

In [ ]:
# ── Filter 2: Female customers from Lahore ────────────────────────────────────
female_lahore = customers[(customers['Gender'] == 'F') & (customers['City'] == 'Lahore')]
print(f"Female customers from Lahore  :  {len(female_lahore)} found")
display(female_lahore[['CustomerID','Name','Gender','City']])

In [ ]:
# ── Filter 3: Customers aged between 25 and 40 ───────────────────────────────
age_25_40 = customers[(customers['Age'] >= 25) & (customers['Age'] <= 40)]
print(f"Customers aged 25–40  :  {len(age_25_40)} found")
display(age_25_40[['CustomerID','Name','Age']])

In [ ]:
# ── Filter 4: Purchases above 50,000 ─────────────────────────────────────────
big_purchases = purchases[purchases['Amount'] > 50000]
print(f"Purchases > 50,000  :  {len(big_purchases)} found")
display(big_purchases)

In [ ]:
# ── Filter 5: Customers with more than 3 purchases ───────────────────────────
purchase_counts = purchases.groupby('CustomerID').size().reset_index(name='PurchaseCount')
freq_buyers = purchase_counts[purchase_counts['PurchaseCount'] > 3]
print(f"Customers with > 3 purchases  :  {len(freq_buyers)} found")
display(freq_buyers)

In [ ]:
# ── Grouping 1: Average income by city ───────────────────────────────────────
avg_income_city = customers.groupby('City')['Income'].mean().reset_index()
avg_income_city.columns = ['City','Avg_Income']
avg_income_city.sort_values('Avg_Income', ascending=False, inplace=True)
print("Average Income by City:")
display(avg_income_city)

In [ ]:
# ── Grouping 2: Total sales by city ──────────────────────────────────────────
merged_for_group = customers.merge(purchases, on='CustomerID', how='inner')
total_sales_city = merged_for_group.groupby('City')['Amount'].sum().reset_index()
total_sales_city.columns = ['City','Total_Sales']
total_sales_city.sort_values('Total_Sales', ascending=False, inplace=True)
print("Total Sales by City:")
display(total_sales_city)

In [ ]:
# ── Grouping 3: Total purchases by gender ────────────────────────────────────
total_purch_gender = merged_for_group.groupby('Gender')['Amount'].sum().reset_index()
total_purch_gender.columns = ['Gender','Total_Purchases']
print("Total Purchases by Gender:")
display(total_purch_gender)

In [ ]:
# ── Grouping 4: Average purchase amount by membership type ───────────────────
merged_mem = merged_for_group.merge(membership, on='CustomerID', how='left')
avg_amount_mem = merged_mem.groupby('MembershipType')['Amount'].mean().reset_index()
avg_amount_mem.columns = ['MembershipType','Avg_Purchase_Amount']
avg_amount_mem.sort_values('Avg_Purchase_Amount', ascending=False, inplace=True)
print("Average Purchase Amount by Membership Type:")
display(avg_amount_mem)

In [ ]:
# ── Grouping 5: Number of customers in each city ─────────────────────────────
cust_per_city = customers.groupby('City')['CustomerID'].count().reset_index()
cust_per_city.columns = ['City','CustomerCount']
cust_per_city.sort_values('CustomerCount', ascending=False, inplace=True)
print("Number of Customers per City:")
display(cust_per_city)

---
## 🔗 Section 4 — Dataset Merging

In [ ]:
# ── Merge 1: customers + purchases (LEFT JOIN) ────────────────────────────────
merged_cp = customers.merge(purchases, on='CustomerID', how='left')
print(f"Merge 1 (customers + purchases):")
print(f"  Shape              : {merged_cp.shape}")
print(f"  Matched records    : {merged_cp['PurchaseID'].notna().sum()}")
print(f"  Unmatched records  : {merged_cp['PurchaseID'].isna().sum()}")
print(f"  Nulls after merge  :\n{merged_cp.isnull().sum()[merged_cp.isnull().sum() > 0]}")
display(merged_cp.head())

In [ ]:
# ── Merge 2: merged_cp + membership (LEFT JOIN) ───────────────────────────────
merged_all = merged_cp.merge(membership, on='CustomerID', how='left')
print(f"Merge 2 (merged + membership):")
print(f"  Shape              : {merged_all.shape}")
print(f"  Matched records    : {merged_all['MembershipType'].notna().sum()}")
print(f"  Unmatched records  : {merged_all['MembershipType'].isna().sum()}")
print(f"\n  Nulls after merge  :")
display(merged_all.isnull().sum().to_frame('NullCount'))

print("\n📋 Final Merged DataFrame (first 10 rows):")
display(merged_all.head(10))

---
## ⚙️ Section 5 — Feature Engineering for AI

In [ ]:
df = merged_all.copy()

# ── Feature 1: Age Group ──────────────────────────────────────────────────────
def age_group(age):
    if age <= 30:
        return 'Young'
    elif age <= 50:
        return 'Adult'
    else:
        return 'Senior'

df['AgeGroup'] = df['Age'].apply(age_group)
print("Feature 1 — AgeGroup distribution:")
display(df['AgeGroup'].value_counts().to_frame())

In [ ]:
# ── Feature 2: Income Category ────────────────────────────────────────────────
def income_category(inc):
    if inc < 40000:
        return 'Low'
    elif inc <= 80000:
        return 'Medium'
    else:
        return 'High'

df['IncomeCategory'] = df['Income'].apply(income_category)
print("Feature 2 — IncomeCategory distribution:")
display(df['IncomeCategory'].value_counts().to_frame())

In [ ]:
# ── Feature 3: Spending Score ─────────────────────────────────────────────────
df['SpendingScore'] = df['Amount'].fillna(0) * df['Quantity'].fillna(0)
print("Feature 3 — SpendingScore (Amount × Quantity):")
print(df[['CustomerID','Amount','Quantity','SpendingScore']].dropna().head(10).to_string(index=False))

In [ ]:
# ── Feature 4: Customer Value ─────────────────────────────────────────────────
def customer_value(score):
    if score > 100000:
        return 'High Value'
    elif score >= 50000:
        return 'Medium Value'
    else:
        return 'Low Value'

df['CustomerValue'] = df['SpendingScore'].apply(customer_value)
print("Feature 4 — CustomerValue distribution:")
display(df['CustomerValue'].value_counts().to_frame())

In [ ]:
# ── Feature 5: Purchase Frequency ────────────────────────────────────────────
freq = purchases.groupby('CustomerID').size().reset_index(name='PurchaseFrequency')
df = df.merge(freq, on='CustomerID', how='left')
df['PurchaseFrequency'] = df['PurchaseFrequency'].fillna(0)
df['PurchaseFrequency'] = df['PurchaseFrequency'].astype('Int64')
print("Feature 5 — PurchaseFrequency (per customer):")
display(df[['CustomerID','Name','PurchaseFrequency']].drop_duplicates('CustomerID').head(10))

In [ ]:
# ── Feature 6: Membership Duration ───────────────────────────────────────────
today = pd.Timestamp(datetime.today().date())
df['JoinDate'] = pd.to_datetime(df['JoinDate'], errors='coerce')
df['MembershipDuration'] = (today - df['JoinDate']).dt.days
print(f"Feature 6 — MembershipDuration (days from JoinDate to {today.date()}):")
display(df[['CustomerID','JoinDate','MembershipDuration']].dropna().drop_duplicates('CustomerID').head(10))

In [ ]:
# ── Show all newly created features ──────────────────────────────────────────
new_features = ['CustomerID','Name','AgeGroup','IncomeCategory',
                'SpendingScore','CustomerValue','PurchaseFrequency','MembershipDuration']
print("\n📋 All Newly Created Features (first 15 rows):")
display(df[new_features].head(15))

---
## 🤖 Section 6 — Prepare Dataset for Machine Learning

In [ ]:
# ── 6.1  Check remaining missing values ──────────────────────────────────────
print("Remaining Missing Values:")
miss_remaining = df.isnull().sum()
display(miss_remaining[miss_remaining > 0].to_frame('Missing Count'))

# Fill any leftover nulls
df['MembershipType'] = df['MembershipType'].fillna('No Membership')
df['MembershipDuration'].fillna(0,             inplace=True)
df['Amount'] = df['Amount'].fillna(df['Amount'].median())
df['Quantity'] = df['Quantity'].fillna(1)
df['PurchaseID'] = df['PurchaseID'].fillna('No Purchase')
df['Product'] = df['Product'].fillna('None')
df['SpendingScore'] = df['SpendingScore'].fillna(0)
df['CustomerValue'] = df['CustomerValue'].fillna('Low Value')
print("\n✅ All remaining missing values handled.")

In [ ]:
# ── 6.2  Encode Categorical Columns ──────────────────────────────────────────
label_cols = ['Gender','City','AgeGroup','IncomeCategory',
              'CustomerValue','MembershipType']

for col in label_cols:
    df[col + '_Encoded'] = pd.Categorical(df[col]).codes

print("✅ Categorical columns encoded.")
for col in label_cols:
    mapping = dict(enumerate(pd.Categorical(df[col]).categories))
    print(f"   {col:<20}: {mapping}")

In [ ]:
# ── 6.3  Normalise Numerical Columns ─────────────────────────────────────────
scaler = MinMaxScaler()
norm_cols = ['Income','Amount','SpendingScore']

df[[c + '_Norm' for c in norm_cols]] = scaler.fit_transform(df[norm_cols])

print("✅ Normalised columns (MinMax 0–1):")
display(df[['Income','Income_Norm','Amount','Amount_Norm','SpendingScore','SpendingScore_Norm']].head(8))

In [ ]:
# ── 6.4  Remove Irrelevant Columns and Save ───────────────────────────────────
drop_cols = ['Name','PurchaseID',
             'Gender','City','AgeGroup','IncomeCategory',    # original (encoded versions kept)
             'CustomerValue','MembershipType',
             'Income','Amount','SpendingScore',              # raw (normalised versions kept)
             'JoinDate','Product']

ai_df = df.drop(columns=drop_cols, errors='ignore')

# Save
ai_df.to_csv('AI_Ready_Customers.csv', index=False)

print(f"✅ AI_Ready_Customers.csv saved!")
print(f"\n   Final Shape   : {ai_df.shape}")
print(f"   Final Columns :\n   {list(ai_df.columns)}")

In [ ]:
# ── 6.5  Sample 20 rows of final dataset ─────────────────────────────────────
print("Sample 20 rows of AI_Ready_Customers:")
display(ai_df.sample(min(20, len(ai_df)), random_state=1))

---
## 📊 Section 7 — Advanced Analytics & Dashboard

In [ ]:
# ── Task 1: Top 10 highest-spending customers ─────────────────────────────────
top_spenders = (
    df.groupby('CustomerID')['SpendingScore_Norm']
      .max()
      .reset_index()
      .merge(customers[['CustomerID','Name']], on='CustomerID')
      .sort_values('SpendingScore_Norm', ascending=False)
      .head(10)
)
print("Top 10 Highest-Spending Customers:")
display(top_spenders)

In [ ]:
# ── Task 2: Customers who never purchased anything ────────────────────────────
buyers = set(purchases['CustomerID'])
non_buyers = customers[~customers['CustomerID'].isin(buyers)]
print(f"Customers who never made a purchase: {len(non_buyers)}")
display(non_buyers[['CustomerID','Name','City','Income']])

In [ ]:
# ── Task 3: City with highest revenue ────────────────────────────────────────
city_revenue = merged_for_group.groupby('City')['Amount'].sum().reset_index()
city_revenue.columns = ['City','Revenue']
city_revenue.sort_values('Revenue', ascending=False, inplace=True)
top_city = city_revenue.iloc[0]
print(f"City with Highest Revenue: {top_city['City']}  (PKR {top_city['Revenue']:,.0f})")
display(city_revenue)

In [ ]:
# ── Task 4: Most popular product ─────────────────────────────────────────────
popular = purchases['Product'].value_counts().reset_index()
popular.columns = ['Product','Count']
print(f"Most Popular Product: {popular.iloc[0]['Product']}  (purchased {popular.iloc[0]['Count']} times)")
display(popular)

In [ ]:
# ── Task 5: Customer ranking based on Spending Score ─────────────────────────
spend_rank = (
    df.groupby('CustomerID')['SpendingScore_Norm'].max()
      .reset_index()
      .merge(customers[['CustomerID','Name']], on='CustomerID')
      .sort_values('SpendingScore_Norm', ascending=False)
      .reset_index(drop=True)
)
spend_rank.index += 1
spend_rank.index.name = 'Rank'
print("Customer Ranking by Spending Score:")
display(spend_rank)

In [ ]:
# ── Task 6: Dashboard Summary Table ──────────────────────────────────────────
dashboard = pd.DataFrame({
    'Metric'  : [
        'Total Customers',
        'Total Purchases',
        'Total Revenue (PKR)',
        'Average Customer Age',
        'Average Income (PKR)',
        'Average Purchase Amount (PKR)',
        'Most Popular City',
        'Highest Revenue City',
        'Most Popular Product',
        'Customers with Membership',
        'Customers without Membership',
        'High Value Customers',
        'Medium Value Customers',
        'Low Value Customers',
    ],
    'Value'   : [
        len(customers),
        len(purchases),
        f"PKR {purchases['Amount'].sum():,.0f}",
        f"{customers['Age'].mean():.1f} yrs",
        f"PKR {customers['Income'].mean():,.0f}",
        f"PKR {purchases['Amount'].mean():,.0f}",
        customers['City'].value_counts().idxmax(),
        city_revenue.iloc[0]['City'],
        popular.iloc[0]['Product'],
        df[df['MembershipType_Encoded'] > 0]['CustomerID'].nunique(),
        df[df['MembershipType_Encoded'] == 0]['CustomerID'].nunique(),
        (df['CustomerValue_Encoded'] == df[df['CustomerValue']=='High Value']['CustomerValue_Encoded'].max()).sum() if 'CustomerValue' in df else 'N/A',
        (df['CustomerValue'] == 'Medium Value').sum(),
        (df['CustomerValue'] == 'Low Value').sum(),
    ]
})
print("📊 DASHBOARD SUMMARY TABLE")
print("=" * 50)
display(dashboard)

---
## ❓ Section 8 — Conceptual Questions & Answers

### Q1. Why is data cleaning important before AI model training?
Raw datasets contain missing values, duplicates, inconsistencies, and outliers. If these are fed directly into a model, the algorithm learns incorrect patterns. For example, a missing Age value filled with zero (instead of median) would make the model think a customer is 0 years old, completely distorting age-based predictions. Clean data ensures the model learns real signal, not noise.

---

### Q2. How do missing values affect machine learning accuracy?
Most machine learning algorithms (Decision Trees, SVM, Logistic Regression) cannot handle NaN values and will throw errors or silently produce wrong results. Even algorithms that tolerate NaN values may assign them incorrectly. Improperly handled missing data introduces bias — for example, if only low-income customers have missing income, mean-imputation would over-estimate their income and skew classification boundaries.

---

### Q3. Why is feature engineering critical in AI projects?
Raw columns rarely capture the full business meaning. For example, `Amount` and `Quantity` individually are less informative than `SpendingScore = Amount × Quantity`. Feature engineering translates domain knowledge into numeric signals the model can exploit, often improving accuracy more than tuning hyperparameters. Features like `AgeGroup`, `IncomeCategory`, and `MembershipDuration` help the model generalize better across unseen customers.

---

### Q4. What problems occur when datasets are merged incorrectly?
- **Wrong join type** (INNER instead of LEFT): customers without purchases are silently dropped, causing the model to never see non-buying behaviour.
- **Duplicate key explosion**: a many-to-many merge creates far more rows than expected, inflating purchase records and misleading aggregations.
- **Key mismatch** (e.g., CustomerID as int in one file, string in another): records fail to join, generating unexpected NaN columns.
- **Data leakage**: joining future data (e.g., 2025 membership info into 2023 training data) inflates model performance falsely.

---

### Q5. How can poor-quality data lead to biased AI predictions?
If the training dataset under-represents certain groups (e.g., female customers or customers from smaller cities), the model learns to perform well only for majority groups. Inconsistent city names (`lahore` vs `Lahore` vs `LAHORE`) can split one city into three groups, making the model treat them as different locations. Outliers in income can cause the model to draw decision boundaries that don't generalise to normal customers. In all cases, poor data quality is directly proportional to biased, unfair, and unreliable predictions.

---
## ✅ Lab Complete

| Deliverable | Status |
|---|---|
| customers.csv loaded & explored | ✅ |
| purchases.csv loaded & explored | ✅ |
| membership.xlsx loaded & explored | ✅ |
| Missing values handled | ✅ |
| Duplicates removed | ✅ |
| Data standardised | ✅ |
| Outliers detected & handled | ✅ |
| Filtering & grouping performed | ✅ |
| Datasets merged (2 merges) | ✅ |
| 6 new features engineered | ✅ |
| Categorical encoding done | ✅ |
| Normalisation applied | ✅ |
| AI_Ready_Customers.csv saved | ✅ |
| Advanced analytics & dashboard | ✅ |
| Conceptual questions answered | ✅ |